# U-Net Training — Medical Image Segmentation
**Projet : Segmentation sémantique d'images médicales**  
**Étudiant : Alaaeddine Bouchamla — Master 1 Génie Télécom, ENISO**  
**Partenaire : Centre de Radiologie Emilie, Libreville, Gabon**

---

This notebook trains a U-Net on the **BraTS 2023** brain tumor segmentation dataset,
then evaluates on a held-out test split. Trained weights are saved and downloaded
for use in the local inference system.

**Runtime**: Google Colab T4 GPU (~2-3 hours for full training)

**Metrics reported**: IoU, Dice Coefficient, Precision, Recall per class

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install -q nibabel monai[all] torch torchvision tqdm matplotlib scikit-learn

In [ ]:
# ── 2. Imports + Mount Google Drive ────────────────────────────────────────
import os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import jaccard_score
from tqdm import tqdm
from pathlib import Path

# Mount Drive — only weights and checkpoints go here (not the dataset)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/radiology_ai'
os.makedirs(DRIVE_DIR, exist_ok=True)

# BraTS data: local /content/ (re-downloads in ~5 min each session, saves Drive storage)
DATA_DIR  = '/content/brats_data'

# Weights + checkpoint: Drive only (small files, survive session wipes)
SAVE_PATH = f'{DRIVE_DIR}/unet_weights.pth'
CKPT_PATH = f'{DRIVE_DIR}/unet_checkpoint.pth'

os.makedirs(DATA_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Drive dir: {DRIVE_DIR}')
print(f'BraTS data dir: {DATA_DIR}  (local, re-downloaded each session)')
ckpt_exists = os.path.exists(CKPT_PATH)
print(f'Checkpoint on Drive: {ckpt_exists}')


In [ ]:
# ── 3. Load BraTS 2023 dataset (downloads to /content/ each session, ~5 min) ─
from monai.apps import DecathlonDataset
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd,
    RandFlipd, RandRotate90d, ToTensord, Resized, NormalizeIntensityd
)

train_transforms = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True),
    Resized(keys=['image', 'label'], spatial_size=(128, 128, 64), mode=['trilinear', 'nearest']),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=['image', 'label'], prob=0.3),
    ToTensord(keys=['image', 'label']),
])
val_transforms = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True),
    Resized(keys=['image', 'label'], spatial_size=(128, 128, 64), mode=['trilinear', 'nearest']),
    ToTensord(keys=['image', 'label']),
])

train_ds = DecathlonDataset(root_dir=DATA_DIR, task='Task01_BrainTumour',
                             transform=train_transforms, section='training',
                             download=True, cache_rate=0.1)
val_ds   = DecathlonDataset(root_dir=DATA_DIR, task='Task01_BrainTumour',
                             transform=val_transforms,   section='validation',
                             download=True, cache_rate=0.1)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=2)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print('Dataset ready.')


In [ ]:
# ── 4. U-Net Architecture ──────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        dh, dw = x2.size(2)-x1.size(2), x2.size(3)-x1.size(3)
        x1 = F.pad(x1, [dw//2, dw-dw//2, dh//2, dh-dh//2])
        return self.conv(torch.cat([x2, x1], dim=1))

class UNet(nn.Module):
    def __init__(self, in_channels=4, num_classes=4, features=[64,128,256,512]):
        super().__init__()
        self.inc = DoubleConv(in_channels, features[0])
        self.downs = nn.ModuleList([Down(features[i], features[i+1]) for i in range(len(features)-1)])
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.ups = nn.ModuleList()
        ch = features[-1]*2
        for f in reversed(features):
            self.ups.append(Up(ch, f))
            ch = f
        self.outc = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skips = [self.inc(x)]
        for d in self.downs: skips.append(d(skips[-1]))
        x = self.bottleneck(skips[-1])
        for up, skip in zip(self.ups, reversed(skips)): x = up(x, skip)
        return self.outc(x)

model = UNet(in_channels=4, num_classes=4).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'U-Net: {total_params:,} parameters ({total_params/1e6:.1f}M)')

In [ ]:
# ── 5. Loss Functions (Dice + Cross-Entropy combined) ─────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, num_classes=logits.shape[1]).permute(0,3,1,2).float()
        dims = (0, 2, 3)
        intersection = (probs * targets_oh).sum(dims)
        cardinality = (probs + targets_oh).sum(dims)
        dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.dice = DiceLoss()
        self.ce = nn.CrossEntropyLoss()
        self.alpha = alpha

    def forward(self, logits, targets):
        return self.alpha * self.dice(logits, targets) + (1-self.alpha) * self.ce(logits, targets)

criterion = CombinedLoss(alpha=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [ ]:
# ── 6. Metrics ─────────────────────────────────────────────────────────────
def compute_metrics(pred, target, num_classes=4):
    pred_flat = pred.cpu().numpy().flatten()
    target_flat = target.cpu().numpy().flatten()
    
    iou = jaccard_score(target_flat, pred_flat, average='macro', zero_division=0)
    
    # Dice per class
    dice_scores = []
    for c in range(num_classes):
        pred_c = (pred_flat == c)
        target_c = (target_flat == c)
        inter = (pred_c & target_c).sum()
        denom = pred_c.sum() + target_c.sum()
        dice_scores.append((2*inter / denom) if denom > 0 else 1.0)
    
    return {'iou': iou, 'dice': np.mean(dice_scores), 'dice_per_class': dice_scores}

print('Metrics function ready.')

In [ ]:
# ── 7. Training Loop (resumes automatically from Drive checkpoint) ──────────
# DRIVE_DIR, SAVE_PATH, CKPT_PATH are defined in cell 2
NUM_EPOCHS = 50
SAVE_EVERY = 5

best_dice   = 0.0
history     = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': []}
start_epoch = 1

# Resume if checkpoint exists on Drive
if os.path.exists(CKPT_PATH):
    print(f'Checkpoint found — resuming...')
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    best_dice   = ckpt['best_dice']
    history     = ckpt['history']
    start_epoch = ckpt['epoch'] + 1
    print(f'  Resumed from epoch {ckpt["epoch"]} | Best Dice: {best_dice:.4f}')
else:
    print('No checkpoint — starting from scratch.')

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Train', leave=False):
        imgs   = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE).long().squeeze(1)
        D = imgs.shape[-1]
        loss_batch = 0
        for s in range(0, D, 4):
            logits = model(imgs[..., s])
            loss   = criterion(logits, labels[..., s])
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_batch += loss.item()
        total_loss += loss_batch / (D // 4)
    return total_loss / len(loader)

@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, all_metrics = 0, []
    for batch in tqdm(loader, desc='Val', leave=False):
        imgs   = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE).long().squeeze(1)
        s      = imgs.shape[-1] // 2
        logits = model(imgs[..., s])
        total_loss += criterion(logits, labels[..., s]).item()
        all_metrics.append(compute_metrics(logits.argmax(dim=1), labels[..., s]))
    return (total_loss / len(loader),
            np.mean([m['iou']  for m in all_metrics]),
            np.mean([m['dice'] for m in all_metrics]))

print(f'Training epochs {start_epoch}–{NUM_EPOCHS} on {DEVICE}...')
for epoch in range(start_epoch, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_iou, val_dice = val_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_dice)

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train {train_loss:.4f} | Val {val_loss:.4f} | '
          f'IoU {val_iou:.4f} | Dice {val_dice:.4f} | {time.time()-t0:.0f}s')

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  -> Best weights → Drive (Dice={best_dice:.4f})')

    if epoch % SAVE_EVERY == 0:
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'best_dice': best_dice, 'history': history}, CKPT_PATH)
        print(f'  -> Checkpoint → Drive (epoch {epoch})')

print(f'\nDone. Best Dice: {best_dice:.4f} | Weights: {SAVE_PATH}')


In [ ]:
# ── 8. Evaluation on Test Set ──────────────────────────────────────────────
# SAVE_PATH is set in cell 7 (Drive path). Run cell 7 first if starting fresh.
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

test_ds = DecathlonDataset(
    root_dir=DATA_DIR, task='Task01_BrainTumour',
    transform=val_transforms, section='test',
    download=False, cache_rate=0.0
)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

from sklearn.metrics import precision_score, recall_score
all_iou, all_dice, all_precision, all_recall = [], [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        imgs   = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE).long().squeeze(1)
        D = imgs.shape[-1]
        for s in range(0, D, 4):
            pred = model(imgs[..., s]).argmax(dim=1)
            m    = compute_metrics(pred, labels[..., s])
            all_iou.append(m['iou'])
            all_dice.append(m['dice'])
            p_flat = pred.cpu().numpy().flatten()
            t_flat = labels[..., s].cpu().numpy().flatten()
            all_precision.append(precision_score(t_flat, p_flat, average='macro', zero_division=0))
            all_recall.append(recall_score(t_flat, p_flat, average='macro', zero_division=0))

print('\n=== TEST RESULTS (copy these into the article) ===')
print(f'Mean IoU:       {np.mean(all_iou):.4f} ± {np.std(all_iou):.4f}')
print(f'Mean Dice:      {np.mean(all_dice):.4f} ± {np.std(all_dice):.4f}')
print(f'Mean Precision: {np.mean(all_precision):.4f} ± {np.std(all_precision):.4f}')
print(f'Mean Recall:    {np.mean(all_recall):.4f} ± {np.std(all_recall):.4f}')


In [ ]:
# ── 9. Plot Training Curves (for the paper) ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='#1a3c6e')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='#c0392b')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_iou'], color='#1a3c6e')
axes[1].axhline(y=0.85, color='green', linestyle='--', label='Target 85%')
axes[1].set_title('IoU (validation)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['val_dice'], color='#c0392b')
axes[2].axhline(y=0.85, color='green', linestyle='--', label='Target 85%')
axes[2].set_title('Dice Coefficient'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('U-Net Training — BraTS 2023 | Bouchamla, ENISO 2025-2026', fontsize=12)
plt.tight_layout()
out = f'{DRIVE_DIR}/training_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')


In [ ]:
# ── 10. Visualise Predictions (for the paper) ──────────────────────────────
CLASS_COLORS = np.array([[0,0,0],[255,0,0],[0,255,0],[255,255,0]], dtype=np.uint8)
CLASS_NAMES = ['Background', 'Necrosis', 'Oedème', 'Tumeur active']

model.eval()
batch = next(iter(test_loader))
imgs = batch['image'].to(DEVICE)
labels = batch['label'].to(DEVICE).long().squeeze(1)
D = imgs.shape[-1]
s = D // 2  # middle slice

with torch.no_grad():
    pred = model(imgs[..., s]).argmax(dim=1)[0].cpu().numpy()

gt = labels[0, ..., s].cpu().numpy() if labels.dim() == 4 else labels[0].cpu().numpy()
img_t1 = imgs[0, 0, ..., s].cpu().numpy()  # T1 channel

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_t1, cmap='gray'); axes[0].set_title('Image IRM (T1)'); axes[0].axis('off')
axes[1].imshow(CLASS_COLORS[gt]); axes[1].set_title('Masque Ground Truth'); axes[1].axis('off')
axes[2].imshow(CLASS_COLORS[pred]); axes[2].set_title('Prédiction U-Net'); axes[2].axis('off')

# Legend
from matplotlib.patches import Patch
legend = [Patch(color=np.array(c)/255, label=n) for c, n in zip(CLASS_COLORS[1:], CLASS_NAMES[1:])]
axes[2].legend(handles=legend, loc='lower right', fontsize=8)

plt.suptitle('Segmentation sémantique U-Net — BraTS 2023', fontsize=12)
plt.tight_layout()
plt.savefig('/content/segmentation_example.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: segmentation_example.png')

In [ ]:
# ── 11. Download weights (already on Drive — this just downloads to your PC) ─
from google.colab import files
files.download(SAVE_PATH)                                    # unet_weights.pth
files.download(f'{DRIVE_DIR}/training_curves.png')
files.download(f'{DRIVE_DIR}/segmentation_example.png')
print('Place unet_weights.pth in the /models/ directory of the project.')


---
## Part 2 — Transfer Learning on CRE Clinical Data (MedSAM Pseudo-Labels)

After pre-training on BraTS 2023, we fine-tune the U-Net on real clinical DICOM slices from the **Centre de Radiologie Émilie (CRE)**.  
Since the CRE data has no manual annotations, we use **MedSAM** (Segment Anything Model fine-tuned on medical images) to generate pseudo-labels automatically.

**Pipeline:**
1. Upload your `pseudo_labels/` folder to Colab (or mount Google Drive)
2. Load the `manifest.json` generated by `backend/ml/medsam/pseudo_labeler.py`
3. Fine-tune the pre-trained U-Net for 20 epochs on the CRE slices
4. Evaluate and download the fine-tuned weights (`unet_cre_finetuned.pth`)


In [ ]:
# ── 12. Upload pseudo-labels (or mount Drive) ──────────────────────────────
import os, json
from pathlib import Path

# Option A: Upload pseudo_labels.zip produced by pseudo_labeler.py
from google.colab import files

print("Upload your pseudo_labels.zip (produced by backend/ml/medsam/pseudo_labeler.py)")
print("Or skip this cell and mount Google Drive below.\n")

uploaded = files.upload()  # select pseudo_labels.zip

if uploaded:
    import zipfile
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall('/content/pseudo_labels')
    print(f"Extracted to /content/pseudo_labels")
else:
    print("No file uploaded.")

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# PSEUDO_LABEL_ROOT = Path('/content/drive/MyDrive/pseudo_labels')


In [ ]:
# ── 13. CRE Dataset (from pseudo-label manifest) ───────────────────────────
import json, random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

PSEUDO_LABEL_ROOT = Path('/content/pseudo_labels')

# Collect manifest entries across all patients
all_entries = []
for manifest_file in PSEUDO_LABEL_ROOT.rglob('manifest.json'):
    with open(manifest_file) as f:
        all_entries.extend(json.load(f))

print(f"Total pseudo-labeled slice/mask pairs: {len(all_entries)}")

class CREDataset(Dataset):
    """
    Loads (slice, mask) pairs saved by pseudo_labeler.py.
    slice: float32 [0,1] array (H, W)
    mask:  uint8 {0,1} array (H, W)  — binary: background / anatomy+pathology
    Returns: image (1, 512, 512), label (512, 512) LongTensor
    """
    TARGET_SIZE = 512

    def __init__(self, entries: list[dict], augment: bool = False):
        self.entries = entries
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        img = np.load(entry['slice']).astype(np.float32)   # (H, W) in [0,1]
        msk = np.load(entry['mask']).astype(np.int64)       # (H, W) in {0,1}

        # Resize to 512x512 if needed
        if img.shape != (self.TARGET_SIZE, self.TARGET_SIZE):
            import cv2
            img = cv2.resize(img, (self.TARGET_SIZE, self.TARGET_SIZE), interpolation=cv2.INTER_LINEAR)
            msk = cv2.resize(msk.astype(np.uint8), (self.TARGET_SIZE, self.TARGET_SIZE),
                             interpolation=cv2.INTER_NEAREST).astype(np.int64)

        img_t = torch.from_numpy(img).unsqueeze(0)   # (1, H, W)
        msk_t = torch.from_numpy(msk)                # (H, W)

        # Basic augmentations
        if self.augment and random.random() > 0.5:
            img_t = torch.flip(img_t, dims=[2])
            msk_t = torch.flip(msk_t, dims=[1])
        if self.augment and random.random() > 0.5:
            img_t = torch.flip(img_t, dims=[1])
            msk_t = torch.flip(msk_t, dims=[0])

        return img_t, msk_t

# 80/20 train/val split
random.shuffle(all_entries)
n_train = int(0.8 * len(all_entries))
train_entries = all_entries[:n_train]
val_entries = all_entries[n_train:]

cre_train = CREDataset(train_entries, augment=True)
cre_val = CREDataset(val_entries, augment=False)
cre_train_loader = DataLoader(cre_train, batch_size=4, shuffle=True, num_workers=2)
cre_val_loader = DataLoader(cre_val, batch_size=2, shuffle=False, num_workers=2)

print(f"CRE train: {len(cre_train)} | CRE val: {len(cre_val)}")
print(f"Train batches: {len(cre_train_loader)} | Val batches: {len(cre_val_loader)}")


In [ ]:
# ── 14. Fine-tune U-Net on CRE data ────────────────────────────────────────
# Load the BraTS-pretrained weights and adapt the input/output heads for
# single-channel CT input and binary segmentation (background / foreground).

import torch
import torch.nn as nn

FINETUNE_PATH = '/content/unet_cre_finetuned.pth'
FINETUNE_EPOCHS = 20
LR_FINETUNE = 5e-5   # lower LR to preserve pretrained features

# Adapt model: swap input conv (4ch→1ch) and output conv (4cls→2cls)
# We do this by replacing only those layers; all encoder weights are kept.
def adapt_unet_for_cre(pretrained_model: UNet) -> UNet:
    cre_model = UNet(in_channels=1, num_classes=2, features=[64,128,256,512]).to(DEVICE)

    # Copy all encoder/decoder weights that match in shape
    pretrained_sd = pretrained_model.state_dict()
    cre_sd = cre_model.state_dict()
    matched = 0
    for k in cre_sd:
        if k in pretrained_sd and cre_sd[k].shape == pretrained_sd[k].shape:
            cre_sd[k] = pretrained_sd[k]
            matched += 1
    cre_model.load_state_dict(cre_sd)
    print(f"Transferred {matched}/{len(cre_sd)} weight tensors from pretrained U-Net")
    return cre_model

cre_model = adapt_unet_for_cre(model)  # model = pretrained BraTS U-Net from cell 7

# Binary combined loss (2 classes: background=0, foreground=1)
finetune_criterion = CombinedLoss(alpha=0.5)
finetune_optimizer = torch.optim.AdamW(cre_model.parameters(), lr=LR_FINETUNE, weight_decay=1e-5)
finetune_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(finetune_optimizer, T_max=FINETUNE_EPOCHS)

def finetune_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for imgs, masks in tqdm(loader, desc='Finetune', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        loss = criterion(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def finetune_val(model, loader):
    model.eval()
    iou_list, dice_list = [], []
    for imgs, masks in tqdm(loader, desc='Val', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        pred = model(imgs).argmax(dim=1)
        m = compute_metrics(pred, masks, num_classes=2)
        iou_list.append(m['iou'])
        dice_list.append(m['dice'])
    return np.mean(iou_list), np.mean(dice_list)

print(f"Fine-tuning on CRE data for {FINETUNE_EPOCHS} epochs...")
best_cre_dice = 0.0
cre_history = {'loss': [], 'iou': [], 'dice': []}

for epoch in range(1, FINETUNE_EPOCHS + 1):
    t0 = time.time()
    loss = finetune_epoch(cre_model, cre_train_loader, finetune_criterion, finetune_optimizer)
    iou, dice = finetune_val(cre_model, cre_val_loader)
    finetune_scheduler.step()

    cre_history['loss'].append(loss)
    cre_history['iou'].append(iou)
    cre_history['dice'].append(dice)

    print(f"Epoch {epoch:2d}/{FINETUNE_EPOCHS} | Loss: {loss:.4f} | IoU: {iou:.4f} | Dice: {dice:.4f} | {time.time()-t0:.0f}s")

    if dice > best_cre_dice:
        best_cre_dice = dice
        torch.save(cre_model.state_dict(), FINETUNE_PATH)
        print(f"  -> Best CRE model saved (Dice={best_cre_dice:.4f})")

print(f"\nFine-tuning complete. Best Dice on CRE: {best_cre_dice:.4f}")


In [ ]:
# ── 15. Plot fine-tuning curves + download weights ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ft_epochs = range(1, len(cre_history['loss']) + 1)

axes[0].plot(ft_epochs, cre_history['loss'], color='#1a3c6e')
axes[0].set_title('Fine-tuning Loss (CRE)'); axes[0].grid(True, alpha=0.3)

axes[1].plot(ft_epochs, cre_history['iou'], label='IoU', color='#1a3c6e')
axes[1].plot(ft_epochs, cre_history['dice'], label='Dice', color='#c0392b')
axes[1].set_title('Fine-tuning Metrics (CRE)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Transfer Learning: BraTS→CRE | Best Dice={best_cre_dice:.4f}', fontsize=12)
plt.tight_layout()
plt.savefig('/content/finetune_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"""
=== FINE-TUNING RESULTS ===
Best Dice (CRE pseudo-labels): {best_cre_dice:.4f}
Weights saved: {FINETUNE_PATH}

NEXT STEPS:
1. Download unet_cre_finetuned.pth → place in /models/
2. Update backend/ml/unet/inference.py WEIGHTS_PATH to use unet_cre_finetuned.pth
3. Run the MedSAM labeler on more patients to improve pseudo-label quality
""")

files.download(FINETUNE_PATH)
files.download('/content/finetune_curves.png')
